# 21: Parsing Files

The fist part of this notebook works with a FASTA file of the SARS-Cov-2 reference genome. The source is here: https://www.ncbi.nlm.nih.gov/nuccore/NC_045512

You can download the file `covid-ref.fsa` from the course repository at: 
https://github.com/Bio724D/Bio724D_2023_2024/tree/main/data

## Reading and writing from files

The `open()` function is the standard way to gain access to a file on your filesystem. It returns a file object, which can be assigned to an identifier.   
   
The primary argument to the `open()` function is a string giving the path and name of the file you want to open. Files can be opened for reading only, writing, or both reading and writing. The default setting is to open the file in read-only mode, or `'r'`. The code below makes this explicit. Other modes are: write-only `'w'`, append `'a'`, and read and write `'r+'`.

In [1]:
# open the file in read-only mode and assign
f = open('covid-ref.fasta', mode = 'r')  

Now that we have a file object, we can use methods like `read()` and `write()` to interact with it. You can read more about `open()` [**here**](https://docs.python.org/3/library/functions.html#open) and about file objects [**here**](https://docs.python.org/3/glossary.html#term-file-object). 

In [2]:
# what class is a file object?
type(f)

_io.TextIOWrapper

The simplest way to manipulate a file is simply to read all the information from it and return the data in the file as a string. The `read()` function performs this for you. Note that it reads *all* the data into your computer's memory. If the input was a very large file this could be a problem.

In [3]:
# reading everything from a file as a single string
s = f.read()

In [4]:
# what kind of object is s?
type(s)

str

In [5]:
# take a look a the first 300 characters
s[:300]

'>NC_045512.2 Severe acute respiratory syndrome coronavirus 2 isolate Wuhan-Hu-1, complete genome\nATTAAAGGTTTATACCTTCCCAGGTAACAAACCAACCAACTTTCGATCTCTTGTAGATCTGTTCTCTAAA\nCGAACTTTAAAATCTGTGTGGCTGTCACTCGGCTGCATGCTTAGTGCACTCACGCAGTATAATTAATAAC\nTAATTACTGTCGTTGACAGGACACGAGTAACTCGTCTATCTTCTGCAGGCTGCTTACGGTT'

In [6]:
# how long is the entire string
len(s)

30428

Once you've read what you need from the file it's good practice to close it (failing to close a file can lead to a memory leak in some contexts, but it's usually not a problem in an interactive environment like Jupyter notebooks).

In [7]:
f.close()

A safer way to read a file is within a `with` statement as illustrated below.  The advantage of the `with` statement is it insures the file is closed (i.e. you don't need to explicitly call the `close()` method) at the end of the `with` block. This is an example of a context manager, an object that guarantees that specific resources are setup *and* closed down, even if errors occur during a program's execution.

In [8]:
with open("covid-ref.fasta") as f:
    s = f.read()

Once we've read the file into a string we can apply all the standard string methods and operators to it:

In [9]:
# how many characters? (compare to above)
len(s)  

30428

In [10]:
# view the first 50 chars
s[:50] 

'>NC_045512.2 Severe acute respiratory syndrome cor'

In [11]:
# and the last 50 characters
s[-50:]

'TCTTAGGAGAATGACAAAAAAAAAAAAAAAAAAAA\nAAAAAAAAAAAAA\n'

## Reading a file by lines

Sometimes it's more convenient or more efficient to get the information in the file in terms of lines.  The `readlines()` method associated with file object read's all the lines at once in a list:

In [12]:
## return a list of the lines in the file
with open("./covid-ref.fasta") as f:
    lines = f.readlines()

In [124]:
type(lines)

list

In [13]:
len(lines)

429

The first line of an entry in a FASTA file is a "header", followed by 1 or more lines of sequence. Header lines start with a `>` (right bracket) symbol. The SARS-CoV-2 genome FASTA only contains one entry. 

In [14]:
# show the "header" line for the reference genome
lines[0]

'>NC_045512.2 Severe acute respiratory syndrome coronavirus 2 isolate Wuhan-Hu-1, complete genome\n'

In [15]:
# subsequent lines are the actual sequence data
lines[1]

'ATTAAAGGTTTATACCTTCCCAGGTAACAAACCAACCAACTTTCGATCTCTTGTAGATCTGTTCTCTAAA\n'

Notice the newline character (`\n`) at the end of each line.

## Reading a file one line at a time

The `readlines()` function illustrated above reads all the lines at once. That's works well if your file has a modest number of lines, but for a file with millions of lines or with very long  lines,  `readlines()` might exhaust the memory of your computer.  One way to work around this is to process files line by line, reading only one line at a time. The code below uses a for loop to read each line and extract its length. Only one line at a time is loaded into memory.

In [16]:
# count the number of characters on each line

with open("covid-ref.fasta") as f:
    nchars = []  # create a list to hold characters counts per line
    for line in f:
        nchars.append(len(line)) 

nchars[:10]  # the first 10 counts

[97, 71, 71, 71, 71, 71, 71, 71, 71, 71]

## Iterating through a file, filtering and concatenating lines

Here we illustrate the process of iterating through the lines of a file, doing some simple filtering, and concatenating lines into a single string. First, we tally the number of each base within the genome. Second, we concatenate the sequence, which in the `.fasta` file contains numerous newline characters.   

Some methods used in these code blocks:   
`items()` returns a key, value pair    
`strip()` removes newlines and other whitespace at beginning/end of lines    
`count()` returns a tally of non-overlapping occurrences of a given substring within the string   


In [17]:
# tally the number of bases in the SARS-Cov2 genome

# create an empty dictionary to hold the base tallies
ctdict = {
    'A':0,
    'T':0,
    'C':0,
    'G':0
}

# read lines one at a time
with open ('covid-ref.fasta') as f:
    for line in f:
        if line.startswith('>'):                 # if the line starts with right bracket
            continue                             # skip it (go to next iteration of for loop)
        clean_line = line.strip()                # if the line is not empty
        for (key, val) in ctdict.items():        # iterate over the four items in the dictionary
            ctdict[key] += clean_line.count(key) # count each base and assign to dictionary

In [18]:
# view the tallies
ctdict

{'A': 8954, 'T': 9594, 'C': 5492, 'G': 5863}

In [19]:
# generate a string consisting of the SARS-Cov2 genome

# create an empty string to hold the concatenated sequence
seq = ""

# read lines one at a time
with open("./covid-ref.fasta") as f:
    for line in f:
        if line[0] == ">":                       # if the line starts with right bracket
            continue                             # skip it (i.e. go to next iteration of for loop)
        if len(line.strip()):                    # if the line is not empty
            seq += line.strip()                  # add it to our seq string object

In [20]:
# total length of COVID reference genome nucleotide sequence
# does not include the header line which we filtered out in our for loop above
len(seq)

29903

In [21]:
# first 100 characters in this sequence
seq[:100]

'ATTAAAGGTTTATACCTTCCCAGGTAACAAACCAACCAACTTTCGATCTCTTGTAGATCTGTTCTCTAAACGAACTTTAAAATCTGTGTGGCTGTCACTC'

## Extracting codons  using string slicing

Since the string we're working with represents the Sars-Cov-2 genome, let's extract a subsequence of interest that represents the gene that encodes the Spike protein. Once we've done so we'll generate  codons from that subsequence and translate them to their corresponding amino acids.

The spike protein coordinates from NCBI are given as: 21563..25384 but these are 1-indexed and inclusive of start/end coordinates. To extract the corresponding sequence from our Python string we need to convert these to 0-indexed coordinates and remember that Python indexing is up to but not including end index

In [22]:
# extract the DNA sequence of the gene encoding the spike protein
spike = seq[21562:25384]

In [23]:
# length of sequence we extracted
len(spike)

3822

In [24]:
# first 10 nucleotides
spike[:10]

'ATGTTTGTTT'

In [25]:
# last 10 nucleotides
spike[-10:]

'TTACACATAA'

In [26]:
# length divisible by 3? (should be True for coding sequence)
len(spike) % 3 == 0

True

Next we'll extract the nucleotide triplets that represent the codons of the spike protein. This task is simple in this case because there are no introns to consider and the gene is encoded in the same strand orientation as the data is provided to us (not always the case). If the gene were on the opposite strand, we could use the function we wrote last week to find the reverse complement.

In [27]:
# create a list of the start positions for each codon
codon_starts = range(0, len(spike), 3)

# convert to list() to see what it looks like, because range() returns an iterator
list(codon_starts)[:10]

[0, 3, 6, 9, 12, 15, 18, 21, 24, 27]

In [28]:
# extract codon subsequences by indexing with codon starts and slicing three characters
spike_codons = [spike[i:i+3] for i in codon_starts]

# take a look to make sure it worked as expected
print(spike[:30])
spike_codons[:10]

ATGTTTGTTTTTCTTGTTTTATTGCCACTA


['ATG', 'TTT', 'GTT', 'TTT', 'CTT', 'GTT', 'TTA', 'TTG', 'CCA', 'CTA']

Now that we have codons, the next step is to translate them. To do this, we will use the genetic code from the standard codon table given by NCBI (https://www.ncbi.nlm.nih.gov/Taxonomy/Utils/wprintgc.cgi#SG1). We will arrange this as a dictionary that maps from codons (the keys) to single-letter representations of amino acids (the values).

In [29]:
# Below is the standard code, cut and pasted from the NCBI website

Base1  = "TTTTTTTTTTTTTTTTCCCCCCCCCCCCCCCCAAAAAAAAAAAAAAAAGGGGGGGGGGGGGGGG"
Base2  = "TTTTCCCCAAAAGGGGTTTTCCCCAAAAGGGGTTTTCCCCAAAAGGGGTTTTCCCCAAAAGGGG"
Base3  = "TCAGTCAGTCAGTCAGTCAGTCAGTCAGTCAGTCAGTCAGTCAGTCAGTCAGTCAGTCAGTCAG"
AAs    = "FFLLSSSSYY**CC*WLLLLPPPPHHQQRRRRIIIMTTTTNNKKSSRRVVVVAAAADDEEGGGG"

For example "TTT" (first column) translates to "F" (phenylalanine), "GGG" (last column) translates to "G" (glycine), "TAG" to "*" (stop codon), etc.   

Next, create a dictionary from those strings.

In [30]:
# turn strings into a dictionary

# create an empty dictionary
codon2AA = dict() 

# loop over every column
for i in range(len(AAs)):    
    codon = Base1[i] + Base2[i] + Base3[i]      # create codon string
    codon2AA[codon] = AAs[i]                    # add entry mapping codon to corresponding AA

In [31]:
## test our dictionary
codon2AA["TTT"], codon2AA["GGG"], codon2AA["TAG"]

('F', 'G', '*')

Having set up this dictionary, we can "translate" the codons into a protein using a simple lookup process. 

In [32]:
# translate the spike protein

# create an empty list to hold the AA sequence
spike_AAs = []

# loop over the codons, retrieving the AA
for codon in spike_codons:
    spike_AAs.append(codon2AA[codon])

In [33]:
# show the first 10 AAs
spike_AAs[:10]

['M', 'F', 'V', 'F', 'L', 'V', 'L', 'L', 'P', 'L']

Now, we just need to concatenate the individual strings into a single string. The `join()` method can do this: is combines strings in order, optionally with a separator. In the code below, we use `""` to indicate no separator.

In [34]:
# concatenate into a single string
spike_protein = "".join(spike_AAs)  

# what happens if you write "!".join(spike_AAs) instead?

In [35]:
# look at the first 50 AAs
spike_protein[:50]

'MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSSVLHS'

In [36]:
# and the last 50 AAs
spike_protein[-50:] 

'IAIVMVTIMLCCMTSCCSCLKGCCSCGSCCKFDEDDSEPVLKGVKLHYT*'

## Parsing a FASTA file

The FASTA file format is the most commonly used file format used to represent nucleotide and protein sequence data.  Wikipedia has a good [overview of the FASTA format](https://en.wikipedia.org/wiki/FASTA_format).  

Summary of FASTA format:
 
 * Each file can hold one or more sequence records
 
 * The beginning of each record is delimited by a line called a header, which has a `>` character at the beginning, followed by the name associated with that record (and an optional description). For example `>seq1 Involved in...` would indicate the beginning of a record with the name `seq1` and the description "Involved in...".
 
 * One or more sequence lines follow each header line. Sequence lines are usually wrapped to have length <=80 characters but this is not required. 
     

Below is a simple function that will parse a FASTA file, returning each record as an element in a Python dictionary with the first word in the header line of each record as the key. 

This implementation isn't particularly robust or optimal, but illustrates some key aspects of flow-control in Python and parsing non-tabular data.


In [37]:
def parse_FASTA(fname):

    # open the file in read-only mode
    f = open(fname, 'r')

    # initialize variables
    record_dict = {}       # initialize an empty dictionary to hold records
    recname = ""           # will hold current record name
    seq = ""               # will hold current seq string
    active_record = False  # indicates whether we are currently working on building a record

    # read one line at a time from the file
    for line in f.readlines():    
        line = line.strip()                # strip any whitespace at beginning/end of line

    # for each line, there are three possibilities: empty, new record, or sequence
        if line == "":                     # is this an empty line?
            continue                       # go to next iteration of for loop
 
        if line[0] == ">":                 # is this a new record?
   
            if active_record:              # did we already have an active record?
                record_dict[recname] = seq # if so, add prior active record to dictionary 
            
            recname = line[1:].split()[0]  # name of new record: split on whitespace, take first element
            seq = ""                       # reset variable holding the sequence
            active_record = True           # set flag to indicate we now have an active record
            continue                       # go to  next iteration of for loop
        
        seq += line                        # this must be a sequence line; add it to the sequence
        
    # done reading lines
    if active_record:                      # we might still have an active record
        record_dict[recname] = seq         # if so, add it to the dictionary

    # remember to close the file!
    f.close() 

    # return the completed dictionary
    return record_dict           

To test our `parse_FASTA` function download the [`Spike-protein-aligned.fasta`](https://github.com/Bio724D/Bio724D_2023_2024/tree/main/data/Spike-protein-aligned.fasta) file  to your computer and modify the paths below as necessary to load and parse the sequence records contained in that file.

In [38]:
# load the data
recs = parse_FASTA("./Spike-protein-aligned.fasta")

In [39]:
# examine the type of object we got back
type(recs)

dict

In [40]:
# how many records are there
len(recs)

7

In [41]:
# what are the keys of the dictionary of recrods
list(recs.keys())  

['KF367457.1',
 'AY278488.2',
 'NC_004718.3',
 'MG772933.1',
 'MT040333.1',
 'MN908947.3',
 'MN996532.1']

In [42]:
# get a specific record, and the first 80 AAs
recs["MN996532.1"][:80]

'M-FVFLVL-LPLVSS----QCVNLTTRTQLPPAYTN--SSTRGVYYPDKVFRSSVLHLTQDLFLPFFSNVTWFHAIHVSG'

In [43]:
# print the first 25 positions in the alignment for all the records
for key in recs.keys():
    print(recs[key][:25])

MKLLVLVF-ATLVSSYTIEKCLDFD
M-FIFLLF-LTLTSGSDLDRCTTFD
M-FIFLLF-LTLTSGSDLDRCTTFD
M-LFFLFLQFALVNS----QCVNLT
M-FVFLFV-LPLVSS----QCVNLT
M-FVFLVL-LPLVSS----QCVNLT
M-FVFLVL-LPLVSS----QCVNLT


## Parsing .csv files

The code above illustrates how to parse a simple file format "from scratch". However, Python is a large and well developed ecosystem; many file formats you are likely to encounter (including FASTA) already have robust and thoroughly tested parsing libraries. In this last section, we'll parse `.csv` files.

In [55]:
# take a look at the file we will working with
with open("nytimes-covid-data-us-states_2022-02.csv") as f:
    s = f.read()
s[:300]    

'date,geoid,state,cases,cases_avg,cases_avg_per_100k,deaths,deaths_avg,deaths_avg_per_100k\n2020-01-21,USA-53,Washington,1,0.14,0,0,0,0\n2020-01-22,USA-53,Washington,0,0.14,0,0,0,0\n2020-01-23,USA-53,Washington,0,0.14,0,0,0,0\n2020-01-24,USA-53,Washington,0,0.14,0,0,0,0\n2020-01-24,USA-17,Illinois,1,0.14,'

This is hard to read and process!   

The code below illustrates how to parse comma-delimited or tab-delimited files using the built-in [`csv` module](https://docs.python.org/3/library/csv.html) that is part of Python's standard library.

In [53]:
import csv

The basic parsing tool is the `csv.reader` function which will parse each line in a file:

In [54]:
# read a .csv file into a list of lines

covid_data = []

with open("nytimes-covid-data-us-states_2022-02.csv", "r") as csvfile:
    reader = csv.reader(csvfile, delimiter=",")
    for row in reader:
        covid_data.append(row)

When using `csv.reader` the rows of the CSV file are returned as lists of strings.

In [56]:
covid_data[:3] 

[['date',
  'geoid',
  'state',
  'cases',
  'cases_avg',
  'cases_avg_per_100k',
  'deaths',
  'deaths_avg',
  'deaths_avg_per_100k'],
 ['2020-01-21', 'USA-53', 'Washington', '1', '0.14', '0', '0', '0', '0'],
 ['2020-01-22', 'USA-53', 'Washington', '0', '0.14', '0', '0', '0', '0']]

An alternative "reader" is [`csv.DictReader`](https://docs.python.org/3/library/csv.html#csv.DictReader) which will return a list containing each row as a separate dictionary, with the keys being specified as the first line of the input file (the typical header line).  If your input file has no header line, you can specify a header with the `fieldnames` argument. 

In [158]:
# read a .csv file into a dictionary

covid_data = []

with open("nytimes-covid-data-us-states_2022-02.csv","r") as csvfile:
    reader = csv.DictReader(csvfile, delimiter=",")
    for row in reader:
        covid_data.append(row)

In [161]:
covid_data[:3]

[{'date': '2020-01-21',
  'geoid': 'USA-53',
  'state': 'Washington',
  'cases': '1',
  'cases_avg': '0.14',
  'cases_avg_per_100k': '0',
  'deaths': '0',
  'deaths_avg': '0',
  'deaths_avg_per_100k': '0'},
 {'date': '2020-01-22',
  'geoid': 'USA-53',
  'state': 'Washington',
  'cases': '0',
  'cases_avg': '0.14',
  'cases_avg_per_100k': '0',
  'deaths': '0',
  'deaths_avg': '0',
  'deaths_avg_per_100k': '0'},
 {'date': '2020-01-23',
  'geoid': 'USA-53',
  'state': 'Washington',
  'cases': '0',
  'cases_avg': '0.14',
  'cases_avg_per_100k': '0',
  'deaths': '0',
  'deaths_avg': '0',
  'deaths_avg_per_100k': '0'}]

Notice that `csv.DictReader()` returns each row as a dictionary, with the keys being the fieldnames.   
   
You can now index individual values by their key (like indexing using a column in a dataframe).

In [59]:
# print specific rows and columns
for row in covid_data[:10]:
    print(row["date"], row["state"], row["cases"], row["deaths"])

2020-01-21 Washington 1 0
2020-01-22 Washington 0 0
2020-01-23 Washington 0 0
2020-01-24 Washington 0 0
2020-01-24 Illinois 1 0
2020-01-25 Washington 0 0
2020-01-25 Illinois 0 0
2020-01-25 California 1 0
2020-01-26 Washington 0 0
2020-01-26 Illinois 0 0
